
# Phase 2 — FinBERT 3-D Sentiment Ablation

This notebook is designed to run **top-to-bottom in Google Colab**.

It follows the established project pipeline:

1. Clone/update the capstone repository.
2. Install repository dependencies.
3. Install FinBERT/transformers dependencies.
4. Load the exact Phase 2 parquet.
5. Use the project's existing cleaning and chronological split.
6. Verify the original Phase 2 row/window counts.
7. Generate FinBERT's 3-D `[positive, neutral, negative]` probabilities.
8. Run four LSTM ablations:
   - Price
   - Price + Fundamentals
   - Price + FinBERT Sentiment
   - Price + Fundamentals + FinBERT Sentiment
9. Select on validation MCC/AUC.
10. Evaluate the selected configuration on the held-out test set.
11. Save CSV, JSON and Markdown results.

**Only the text representation changes from the previous 768-D FinBERT experiment.**



## 1. Colab Setup

The first cell clones the repository exactly as a fresh Colab runtime should.

If `/content/capstone` already exists, it pulls the latest `main`.


In [ ]:

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/AdityaMelkote3004/capstone.git"
REPO_DIR = "/content/capstone"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )
else:
    print("Repository already exists; pulling latest main...")
    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only"],
        check=False
    )

os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Working directory:", os.getcwd())
print("✓ Repository ready.")



## 2. Install Dependencies

Use the repository's own `requirements.txt` if present, then add the packages needed for this FinBERT experiment.


In [ ]:

import glob

requirements = [
    "requirements.txt",
    "requirements/requirements.txt",
    "requirements/base.txt",
]

req_found = False

for req in requirements:
    if os.path.exists(req):
        print("Installing:", req)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", req],
            check=True
        )
        req_found = True
        break

if not req_found:
    print("No repository requirements file found.")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers",
        "sentencepiece",
        "accelerate",
        "pyarrow",
        "scikit-learn"
    ],
    check=True
)

print("✓ Dependencies installed.")



## 3. Imports, Seed and Device


In [ ]:

import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score
)

SEED = 42
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("✓ Imports successful.")



## 4. Verify Repository and Locate the Exact Phase 2 Dataset


In [ ]:

print("Repository contents:")
for item in sorted(os.listdir(REPO_DIR))[:50]:
    print(" ", item)

parquets = glob.glob(
    os.path.join(REPO_DIR, "**", "*.parquet"),
    recursive=True
)

phase2 = [
    p for p in parquets
    if os.path.basename(p).lower()
    == "stocknet_final_modeling_set_phase2.parquet"
]

if not phase2:
    raise FileNotFoundError(
        "stocknet_final_modeling_set_phase2.parquet "
        "was not found. Restore the exact Phase 2 parquet first."
    )

PHASE2_PATH = phase2[0]
print("\nUsing:", PHASE2_PATH)



## 5. Import the Existing Project Data Pipeline

We intentionally reuse the project's existing functions rather than creating a new preprocessing pipeline.


In [ ]:

from src.data.stocknet_dataset import (
    load_and_clean,
    split_by_date,
    compute_norm_stats,
    normalize,
)

print("✓ Existing project data utilities imported.")



## 6. Load/Clean and Verify the Original Global Split

Expected rows:

| Split | Rows |
|---|---:|
| Train | 15,969 |
| Validation | 4,359 |
| Test | 6,275 |


In [ ]:

df = load_and_clean(PHASE2_PATH)

print("Cleaned shape:", df.shape)
print("Date range:", df["Date"].min(), "→", df["Date"].max())
print("Tickers:", df["Ticker"].nunique())

if len(df) != 26603:
    raise RuntimeError(
        f"Expected 26,603 cleaned rows, got {len(df)}."
    )

train_df, val_df, test_df = split_by_date(df)

actual = (
    len(train_df),
    len(val_df),
    len(test_df)
)

print("Train rows:", actual[0])
print("Val rows:  ", actual[1])
print("Test rows: ", actual[2])

if actual != (15969, 4359, 6275):
    raise RuntimeError(
        f"Original Phase 2 split mismatch: {actual}"
    )

print("✓ Original split verified.")



## 7. Structured Features

Phase 1 established 14 price features and 8 fundamental features.


In [ ]:

PRICE_FEATURES = [
    "Return", "RSI_14", "MACD", "MACD_Signal", "MACD_Hist",
    "Volatility_5", "Volatility_20",
    "Price_MA5_Ratio", "Price_MA10_Ratio", "Price_MA20_Ratio",
    "Volume_Change", "HL_Spread", "MA_5", "MA_10"
]

FUNDAMENTAL_FEATURES = [
    "Revenue", "NetIncome", "TotalAssets", "TotalLiabilities",
    "StockholdersEquity", "EPS", "Cash", "ROA"
]

for col in PRICE_FEATURES + FUNDAMENTAL_FEATURES:
    if col not in df.columns:
        raise KeyError(f"Missing required feature: {col}")

print("Price:", len(PRICE_FEATURES))
print("Fundamentals:", len(FUNDAMENTAL_FEATURES))



## 8. Load FinBERT Classification Model

We use the classification head:

`logits → softmax → [positive, neutral, negative]`

This is the 3-D representation suggested by your peer.


In [ ]:

FINBERT_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(
    FINBERT_NAME
)

finbert = AutoModelForSequenceClassification.from_pretrained(
    FINBERT_NAME
).to(DEVICE)

finbert.eval()

print("FinBERT labels:", finbert.config.id2label)
print("Number of labels:", finbert.config.num_labels)

if finbert.config.num_labels != 3:
    raise RuntimeError("FinBERT classifier does not have 3 classes.")

print("✓ FinBERT ready.")



## 9. Verify Company Text

The same `Company_Texts` field used for the earlier company-text FinBERT experiment is used here.


In [ ]:

if "Company_Texts" not in df.columns:
    raise KeyError(
        "Company_Texts is missing from the Phase 2 parquet."
    )

TEXT_COLUMN = "Company_Texts"

print("Text column:", TEXT_COLUMN)
print("Non-empty rows:",
      (df[TEXT_COLUMN].fillna("").astype(str).str.strip() != "").sum())



## 10. Generate the 3-D FinBERT Sentiment Representation

Each original row receives exactly three values:

- Positive
- Neutral
- Negative

No rows are reordered.


In [ ]:

texts = (
    df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .tolist()
)

sentiment = np.zeros(
    (len(df), 3),
    dtype=np.float32
)

BATCH_SIZE = 32
MAX_LENGTH = 256

with torch.no_grad():

    for start in range(
        0,
        len(texts),
        BATCH_SIZE
    ):

        batch = texts[start:start + BATCH_SIZE]

        positions = [
            i for i, text in enumerate(batch)
            if text.strip()
        ]

        if positions:

            valid_texts = [
                batch[i] for i in positions
            ]

            inputs = tokenizer(
                valid_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )

            inputs = {
                k: v.to(DEVICE)
                for k, v in inputs.items()
            }

            logits = finbert(**inputs).logits

            probs = torch.softmax(
                logits,
                dim=1
            ).cpu().numpy().astype(np.float32)

            for j, position in enumerate(positions):
                sentiment[
                    start + position
                ] = probs[j]

        if start == 0 or start % 640 == 0:
            print(
                f"{min(start + BATCH_SIZE, len(texts)):,}/"
                f"{len(texts):,}"
            )

if not np.isfinite(sentiment).all():
    raise RuntimeError(
        "FinBERT sentiment contains NaN/Inf values."
    )

print("Sentiment shape:", sentiment.shape)
print("First rows:")
print(sentiment[:5])
print("Probability sums:")
print(sentiment[:5].sum(axis=1))



## 11. Attach Sentiment to the Exact Original Splits


In [ ]:

SENTIMENT_FEATURES = [
    "FinBERT_Positive",
    "FinBERT_Neutral",
    "FinBERT_Negative"
]

df_sent = df.copy()

for i, col in enumerate(SENTIMENT_FEATURES):
    df_sent[col] = sentiment[:, i]

train_sent = df_sent.loc[train_df.index].copy()
val_sent = df_sent.loc[val_df.index].copy()
test_sent = df_sent.loc[test_df.index].copy()

if (
    len(train_sent),
    len(val_sent),
    len(test_sent)
) != (15969, 4359, 6275):
    raise RuntimeError("Sentiment split alignment failed.")

print("✓ Exact row alignment preserved.")



## 12. Normalize Structured Features Using Training Statistics Only


In [ ]:

STRUCTURED_FEATURES = (
    PRICE_FEATURES +
    FUNDAMENTAL_FEATURES
)

means, stds = compute_norm_stats(
    train_sent,
    STRUCTURED_FEATURES
)

train_sent = normalize(
    train_sent.copy(),
    STRUCTURED_FEATURES,
    means,
    stds
)

val_sent = normalize(
    val_sent.copy(),
    STRUCTURED_FEATURES,
    means,
    stds
)

test_sent = normalize(
    test_sent.copy(),
    STRUCTURED_FEATURES,
    means,
    stds
)

print("✓ Train-only normalization complete.")



## 13. Construct the Original 5-Day Windows

Expected:

```text
Train windows: 15,534
Val windows:    3,924
Test windows:   5,840
```

If these do not match, the notebook stops rather than producing incomparable results.


In [ ]:

class StockNetWindowDataset(Dataset):

    def __init__(
        self,
        frame,
        feature_columns,
        window_size=5
    ):
        frame = (
            frame
            .sort_values(["Ticker", "Date"])
            .reset_index(drop=True)
        )

        self.samples = []

        for _, group in frame.groupby(
            "Ticker",
            sort=False
        ):
            group = (
                group
                .sort_values("Date")
                .reset_index(drop=True)
            )

            for i in range(
                window_size,
                len(group)
            ):
                x = group.iloc[
                    i-window_size:i
                ][feature_columns].to_numpy(
                    dtype=np.float32
                )

                y = int(group.iloc[i]["Target"])

                self.samples.append(
                    (
                        torch.tensor(
                            x,
                            dtype=torch.float32
                        ),
                        torch.tensor(
                            y,
                            dtype=torch.long
                        )
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


WINDOW_SIZE = 5

ALL_FEATURES = (
    STRUCTURED_FEATURES +
    SENTIMENT_FEATURES
)

train_full = StockNetWindowDataset(
    train_sent,
    ALL_FEATURES,
    WINDOW_SIZE
)

val_full = StockNetWindowDataset(
    val_sent,
    ALL_FEATURES,
    WINDOW_SIZE
)

test_full = StockNetWindowDataset(
    test_sent,
    ALL_FEATURES,
    WINDOW_SIZE
)

counts = (
    len(train_full),
    len(val_full),
    len(test_full)
)

print("Train windows:", counts[0])
print("Val windows:  ", counts[1])
print("Test windows: ", counts[2])

if counts != (15534, 3924, 5840):
    raise RuntimeError(
        f"Window mismatch. Expected "
        f"(15534, 3924, 5840), got {counts}"
    )

print("✓ Original window counts verified.")



## 14. Four Experiments

| Experiment | Input |
|---|---|
| A | Price |
| B | Price + Fundamentals |
| C | Price + FinBERT Sentiment |
| D | Price + Fundamentals + FinBERT Sentiment |


In [ ]:

EXPERIMENTS = {
    "A_Price":
        PRICE_FEATURES,

    "B_Price_Fundamentals":
        PRICE_FEATURES + FUNDAMENTAL_FEATURES,

    "C_Price_FinBERT_Sentiment":
        PRICE_FEATURES + SENTIMENT_FEATURES,

    "D_Price_Fundamentals_FinBERT_Sentiment":
        PRICE_FEATURES +
        FUNDAMENTAL_FEATURES +
        SENTIMENT_FEATURES
}

for name, cols in EXPERIMENTS.items():
    print(name, "→", len(cols), "features")



## 15. Select Features From the Same Windows


In [ ]:

class SelectedDataset(Dataset):

    def __init__(
        self,
        full_dataset,
        all_columns,
        selected_columns
    ):
        self.full = full_dataset

        self.indices = [
            all_columns.index(col)
            for col in selected_columns
        ]

    def __len__(self):
        return len(self.full)

    def __getitem__(self, index):
        x, y = self.full[index]

        return x[:, self.indices], y


experiment_datasets = {}

for name, columns in EXPERIMENTS.items():

    experiment_datasets[name] = {
        "train": SelectedDataset(
            train_full,
            ALL_FEATURES,
            columns
        ),
        "val": SelectedDataset(
            val_full,
            ALL_FEATURES,
            columns
        ),
        "test": SelectedDataset(
            test_full,
            ALL_FEATURES,
            columns
        )
    }

print("✓ All four datasets ready.")



## 16. LSTM Baseline


In [ ]:

class StockNetLSTM(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])


def evaluate(model, loader):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():

        for x, y in loader:

            logits = model(
                x.to(DEVICE)
            )

            y_true.extend(
                y.numpy()
            )

            y_pred.extend(
                logits.argmax(
                    dim=1
                ).cpu().numpy()
            )

            y_prob.extend(
                torch.softmax(
                    logits,
                    dim=1
                )[:, 1].cpu().numpy()
            )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    return {
        "accuracy": accuracy_score(
            y_true, y_pred
        ),
        "f1": f1_score(
            y_true, y_pred,
            zero_division=0
        ),
        "mcc": matthews_corrcoef(
            y_true, y_pred
        ),
        "auc": roc_auc_score(
            y_true, y_prob
        )
    }



## 17. Training Function

Same main training protocol:

- Adam
- LR = 0.001
- Weight decay = 1e-4
- Batch size = 64
- Max epochs = 50
- Patience = 10
- Early stopping on validation MCC
- Gradient clipping = 1.0
- Seed = 42


In [ ]:

def train_experiment(
    bundle,
    input_dim,
    max_epochs=50,
    patience=10
):

    train_loader = DataLoader(
        bundle["train"],
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        bundle["val"],
        batch_size=64,
        shuffle=False
    )

    test_loader = DataLoader(
        bundle["test"],
        batch_size=64,
        shuffle=False
    )

    model = StockNetLSTM(
        input_dim=input_dim
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    best_mcc = -float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, max_epochs + 1):

        model.train()

        for x, y in train_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

        val_metrics = evaluate(
            model,
            val_loader
        )

        if val_metrics["mcc"] > best_mcc:

            best_mcc = val_metrics["mcc"]

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            wait = 0

        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Ep {epoch:3d} | "
                f"Val Acc={val_metrics['accuracy']:.3f} "
                f"F1={val_metrics['f1']:.3f} "
                f"MCC={val_metrics['mcc']:.4f} "
                f"AUC={val_metrics['auc']:.4f}"
            )

        if wait >= patience:
            print("Early stop at epoch", epoch)
            break

    if best_state is None:
        raise RuntimeError(
            "No best model checkpoint was created."
        )

    model.load_state_dict(best_state)

    return (
        model,
        evaluate(model, val_loader),
        evaluate(model, test_loader)
    )



## 18. Run All Four Models


In [ ]:

results = {}
models = {}

for name, columns in EXPERIMENTS.items():

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    set_seed()

    model, val_metrics, test_metrics = train_experiment(
        experiment_datasets[name],
        input_dim=len(columns)
    )

    models[name] = model

    results[name] = {
        "validation": val_metrics,
        "test": test_metrics
    }

    print("\nValidation:", val_metrics)
    print("Test:", test_metrics)



## 19. Results


In [ ]:

rows = []

for name, result in results.items():

    rows.append({
        "Experiment": name,
        "Val_Accuracy": result["validation"]["accuracy"],
        "Val_F1": result["validation"]["f1"],
        "Val_MCC": result["validation"]["mcc"],
        "Val_AUC": result["validation"]["auc"],
        "Test_Accuracy": result["test"]["accuracy"],
        "Test_F1": result["test"]["f1"],
        "Test_MCC": result["test"]["mcc"],
        "Test_AUC": result["test"]["auc"],
    })

results_df = pd.DataFrame(rows)

results_df = results_df.sort_values(
    ["Val_MCC", "Val_AUC"],
    ascending=False
)

display(results_df)



## 20. Select on Validation, Then Report Test


In [ ]:

ranking = sorted(
    results.items(),
    key=lambda item: (
        item[1]["validation"]["mcc"],
        item[1]["validation"]["auc"]
    ),
    reverse=True
)

for rank, (name, result) in enumerate(
    ranking,
    start=1
):
    v = result["validation"]

    print(
        f"{rank}. {name} | "
        f"MCC={v['mcc']:.4f} | "
        f"AUC={v['auc']:.4f}"
    )

BEST_EXPERIMENT = ranking[0][0]

print("\nSelected:", BEST_EXPERIMENT)

print("\nFINAL TEST RESULT")

for metric, value in results[
    BEST_EXPERIMENT
]["test"].items():

    print(
        f"{metric.upper():>10}: {value:.4f}"
    )



## 21. Previous 768-D FinBERT References

| Feature Set | Accuracy | F1 | MCC | AUC |
|---|---:|---:|---:|---:|
| Price + Company FinBERT | 0.5248 | 0.4721 | 0.0459 | 0.5304 |
| Price + Fundamentals + Company FinBERT | 0.4870 | 0.6550 | 0.0000 | 0.5270 |

The new experiment should be compared against these only after confirming the original row/window counts.



## 22. Save Results + Generate Markdown


In [ ]:

RESULT_DIR = os.path.join(
    REPO_DIR,
    "results",
    "phase2_finbert_sentiment"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

csv_path = os.path.join(
    RESULT_DIR,
    "results.csv"
)

json_path = os.path.join(
    RESULT_DIR,
    "results.json"
)

md_path = os.path.join(
    RESULT_DIR,
    "phase2_finbert_sentiment.md"
)

results_df.to_csv(
    csv_path,
    index=False
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        results,
        f,
        indent=2
    )

report = [
    "# Phase 2 — FinBERT 3-D Sentiment Ablation",
    "",
    "## Objective",
    "",
    "Replace the previous 768-D Company FinBERT representation with FinBERT's three-class sentiment probabilities while retaining the established Phase 2 data protocol.",
    "",
    "## Verified Dataset",
    "",
    "| Quantity | Value |",
    "|---|---:|",
    "| Train rows | 15,969 |",
    "| Validation rows | 4,359 |",
    "| Test rows | 6,275 |",
    "| Train windows | 15,534 |",
    "| Validation windows | 3,924 |",
    "| Test windows | 5,840 |",
    "| Window size | 5 |",
    "",
    "## Representation",
    "",
    "FinBERT classification logits are converted to:",
    "",
    "- Positive probability",
    "- Neutral probability",
    "- Negative probability",
    "",
    "## Experiments",
    "",
    "| Experiment | Input |",
    "|---|---|",
    "| A | Price |",
    "| B | Price + Fundamentals |",
    "| C | Price + FinBERT Sentiment |",
    "| D | Price + Fundamentals + FinBERT Sentiment |",
    "",
    "## Results",
    "",
    results_df.to_markdown(index=False),
    "",
    "## Selected Configuration",
    "",
    f"**{BEST_EXPERIMENT}** selected using validation MCC, with validation AUC as secondary criterion.",
    "",
    "## Final Test Result",
    "",
    "| Metric | Value |"
]

for metric, value in results[
    BEST_EXPERIMENT
]["test"].items():
    report.append(
        f"| {metric.upper()} | {value:.4f} |"
    )

report += [
    "",
    "## Previous 768-D FinBERT References",
    "",
    "| Feature Set | Accuracy | F1 | MCC | AUC |",
    "|---|---:|---:|---:|---:|",
    "| Price + Company FinBERT | 0.5248 | 0.4721 | 0.0459 | 0.5304 |",
    "| Price + Fundamentals + Company FinBERT | 0.4870 | 0.6550 | 0.0000 | 0.5270 |",
    "",
    "## Interpretation",
    "",
    "This is an ablation of the established pipeline. The dataset split, normalization protocol, five-day windows, LSTM training protocol and evaluation procedure are retained while the text representation is reduced from 768 dimensions to three explicit sentiment probabilities."
]

Path(md_path).write_text(
    "\n".join(report),
    encoding="utf-8"
)

print("Saved:")
print(csv_path)
print(json_path)
print(md_path)



## 23. Final Sanity Check

The notebook should show:

```text
Train rows:     15969
Val rows:        4359
Test rows:       6275

Train windows:  15534
Val windows:     3924
Test windows:    5840
```

If any of those numbers are different, **do not use the results**.

The generated artifacts are saved under:

```text
results/phase2_finbert_sentiment/
```


In [ ]:

print("Final verification")
print("=" * 70)

checks = {
    "Phase 2 parquet": PHASE2_PATH,
    "results.csv": csv_path,
    "results.json": json_path,
    "phase2_finbert_sentiment.md": md_path,
}

for label, path in checks.items():
    print(
        ("✓" if os.path.exists(path) else "✗"),
        label,
        "->",
        path
    )

print("\nGit status:")
subprocess.run(
    ["git", "-C", REPO_DIR, "status", "--short"]
)
